In [129]:
import pandas as pd
import numpy as np
import unicodedata

from pathlib import Path
from sklearn.preprocessing import MinMaxScaler

BASE_DIR = Path.cwd().parent

DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_EXTERNAL = BASE_DIR / "data" / "external"

df = pd.read_csv(
    DATA_PROCESSED / "dataset_maestro_granada_v2.csv"
)

print(df.shape)
df.head()

(128, 9)


,codigo_ine,municipio,poblacion_2025,superficie_residencial_m2,n_poligonos,superficie_media_poligono,precio_m2,fuente,anio
0,18905,"Gabias, Las",23584,3.682035e+06,1,3.682035e+06,NaN,NaN,NaN
1,18089,Guadix,18881,3.460928e+06,7,4.944182e+05,NaN,NaN,NaN
2,18022,Atarfe,20914,3.364040e+06,1,3.364040e+06,1369.0,Idealista,2026.0
3,18029,Benamaurel,2235,3.295178e+06,9,3.661309e+05,NaN,NaN,NaN
4,18003,Albolote,19768,2.949836e+06,1,2.949836e+06,NaN,NaN,NaN


In [130]:
df = df.drop(
    columns=[
        "precio_m2",
        "fuente",
        "anio"
    ],
    errors="ignore"
)

df.columns

Index(['codigo_ine', 'municipio', 'poblacion_2025',
       'superficie_residencial_m2', 'n_poligonos',
       'superficie_media_poligono'],
      dtype='object')

In [131]:
precios = pd.read_csv(
    DATA_EXTERNAL / "precios_municipios_granada_2026.csv"
)

print(precios.shape)

precios.head()

(42, 6)


,municipio,comarca,precio_venta_eur_m2,precio_alquiler_eur_m2_mes,fuente,fecha
0,Sierra Nevada,Sierra Nevada,3081,NaN,Idealista,2026-05
1,Almuñécar,Costa Tropical,3067,10.8,Idealista,2026-05
2,Granada Capital,Área Metropolitana,2645,10.7,Idealista,2026-05
3,Salobreña,Costa Tropical,2173,9.1,Idealista,2026-05
4,Armilla,Área Metropolitana,2144,9.0,Idealista,2026-05


In [132]:
def limpiar_texto(texto):

    texto = str(texto).strip().lower()

    texto = ''.join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )

    return texto

In [133]:
df["municipio_merge"] = df["municipio"].apply(limpiar_texto)

precios["municipio_merge"] = (
    precios["municipio"]
    .apply(limpiar_texto)
)

In [134]:
df = df.merge(
    precios[
        [
            "municipio_merge",
            "precio_venta_eur_m2",
            "comarca"
        ]
    ],
    on="municipio_merge",
    how="left"
)

print(
    "Municipios con precio:",
    df["precio_venta_eur_m2"].notna().sum()
)

Municipios con precio: 28


In [135]:
df[
    df["precio_venta_eur_m2"].isna()
][
    ["municipio"]
]

,municipio
0,"Gabias, Las"
3,Benamaurel
13,Pinos Puente
15,Cúllar
16,Dílar
...,...
123,Jete
124,Torrenueva Costa
125,Cáñar
126,Dúdar


In [136]:
df[
    df["precio_venta_eur_m2"].isna()
][
    ["municipio","municipio_merge"]
].sort_values("municipio")

,municipio,municipio_merge
122,Agrón,agron
103,Alamedilla,alamedilla
91,Albondón,albondon
99,Albuñuelas,albunuelas
92,Aldeire,aldeire
...,...,...
69,Villanueva de las Torres,villanueva de las torres
90,Válor,valor
74,Víznar,viznar
48,Zújar,zujar


In [137]:
precios[
    ["municipio","municipio_merge"]
].sort_values("municipio")

,municipio,municipio_merge
15,Albolote,albolote
30,Albuñol,albunol
26,Alfacar,alfacar
38,Alhama de Granada,alhama de granada
7,Alhendín,alhendin
1,Almuñécar,almunecar
4,Armilla,armilla
20,Atarfe,atarfe
40,Baza,baza
24,Belicena,belicena


In [138]:
df[
    df["precio_venta_eur_m2"].notna()
][
    ["municipio","precio_venta_eur_m2"]
].sort_values(
    "precio_venta_eur_m2",
    ascending=False
)

,municipio,precio_venta_eur_m2
12,Almuñécar,3067.0
6,Salobreña,2173.0
9,Armilla,2144.0
11,Churriana de la Vega,1903.0
20,Alhendín,1794.0
19,Maracena,1758.0
24,Cájar,1692.0
32,Cenes de la Vega,1648.0
4,Albolote,1612.0
37,Motril,1491.0


In [139]:
mediana_precio = df["precio_venta_eur_m2"].median()

print("Mediana:", mediana_precio)

Mediana: 1142.0


In [140]:
df["precio_imputado"] = (
    df["precio_venta_eur_m2"].isna()
)

In [141]:
df["precio_venta_eur_m2"] = (
    df["precio_venta_eur_m2"]
    .fillna(mediana_precio)
)

In [142]:
df["precio_venta_eur_m2"].isna().sum()

np.int64(0)

In [143]:
# ==================================================
# IMPUTACIÓN MEDIANA PRECIO VENTA
# ==================================================

df["precio_imputado"] = (
    df["precio_venta_eur_m2"].isna()
)

mediana_precio = df["precio_venta_eur_m2"].median()

print("Mediana provincial:", mediana_precio)

df["precio_venta_eur_m2"] = (
    df["precio_venta_eur_m2"]
    .fillna(mediana_precio)
)

print(
    "Nulos restantes:",
    df["precio_venta_eur_m2"].isna().sum()
)

Mediana provincial: 1142.0
Nulos restantes: 0


In [144]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["score_precio"] = scaler.fit_transform(
    df[["precio_venta_eur_m2"]]
)

In [145]:
df[
    [
        "municipio",
        "precio_venta_eur_m2",
        "score_precio",
        "precio_imputado"
    ]
].head(20)

,municipio,precio_venta_eur_m2,score_precio,precio_imputado
0,"Gabias, Las",1142.0,0.232456,False
1,Guadix,839.0,0.111643,False
2,Atarfe,1380.0,0.327352,False
3,Benamaurel,1142.0,0.232456,False
4,Albolote,1612.0,0.419856,False
5,Loja,772.0,0.084928,False
6,Salobreña,2173.0,0.643541,False
7,Íllora,559.0,0.000000,False
8,Baza,656.0,0.038676,False
9,Armilla,2144.0,0.631978,False


In [146]:
df.columns.tolist()

['codigo_ine',
 'municipio',
 'poblacion_2025',
 'superficie_residencial_m2',
 'n_poligonos',
 'superficie_media_poligono',
 'municipio_merge',
 'precio_venta_eur_m2',
 'comarca',
 'precio_imputado',
 'score_precio']

In [147]:
distancias = pd.read_csv(
    DATA_EXTERNAL / "distancia_pueblos.csv",
    sep=";"
)

distancias.head()

,localidad,Distancia en Kms Capital,Distancia en kms a la costa
0,Agron,36.41,67.9
1,Alamedilla,72.89,106.9
2,Albolote,9.78,71.1
3,Albondon,114.59,43.9
4,Albunan,63.81,41.6


In [148]:
distancias.columns.tolist()

['localidad', 'Distancia en Kms Capital', 'Distancia en kms a la costa']

In [149]:
distancias["municipio_merge"] = (
    distancias["localidad"]
    .apply(limpiar_texto)
)

In [150]:
correcciones = {
    "gabias, las": "las gabias",
    "valle, el": "el valle",
    "malaha, la": "la malaha",
    "calahorra, la": "la calahorra",
    "peza, la": "la peza"
}

df["municipio_merge"] = (
    df["municipio_merge"]
    .replace(correcciones)
)

In [151]:
distancias = distancias.rename(
    columns={
        "Distancia en Kms Capital":
            "distancia_capital_km",

        "Distancia en kms a la costa":
            "distancia_costa_km"
    }
)

In [152]:
df = df.merge(
    distancias[
        [
            "municipio_merge",
            "distancia_capital_km",
            "distancia_costa_km"
        ]
    ],
    on="municipio_merge",
    how="left"
)

In [153]:
print(
    "Sin capital:",
    df["distancia_capital_km"].isna().sum()
)

print(
    "Sin costa:",
    df["distancia_costa_km"].isna().sum()
)

Sin capital: 7
Sin costa: 8


In [154]:
print(
    df["distancia_capital_km"].median()
)

print(
    df["distancia_costa_km"].median()
)

56.02
67.5


In [155]:
df["distancia_capital_km"] = (
    df["distancia_capital_km"]
    .fillna(
        df["distancia_capital_km"].median()
    )
)

df["distancia_costa_km"] = (
    df["distancia_costa_km"]
    .fillna(
        df["distancia_costa_km"].median()
    )
)

In [156]:
print(
    df["distancia_capital_km"].isna().sum()
)

print(
    df["distancia_costa_km"].isna().sum()
)

0
0


In [157]:
df["score_capital_norm"] = 1 - (
    MinMaxScaler().fit_transform(
        df[["distancia_capital_km"]]
    )
)

In [158]:
municipios_litoral = [
    "Almuñécar",
    "Salobreña",
    "Motril",
    "Torrenueva Costa",
    "Vélez de Benaudalla"
]

df["score_litoral"] = 0

df.loc[
    df["municipio"].isin(
        municipios_litoral
    ),
    "score_litoral"
] = 1

In [159]:
df[
    [
        "municipio",
        "distancia_capital_km",
        "score_capital_norm",
        "score_litoral"
    ]
].head(20)

,municipio,distancia_capital_km,score_capital_norm,score_litoral
0,"Gabias, Las",9.05,0.970762,0
1,Guadix,52.61,0.700354,0
2,Atarfe,10.45,0.962071,0
3,Benamaurel,109.41,0.347756,0
4,Albolote,9.78,0.966230,0
5,Loja,53.49,0.694891,0
6,Salobreña,67.63,0.607114,1
7,Íllora,34.55,0.812465,0
8,Baza,96.15,0.430070,0
9,Armilla,8.12,0.976535,0


In [160]:
df["suelo_por_habitante"] = (
    df["superficie_residencial_m2"]
    /
    df["poblacion_2025"]
)

In [161]:
df["log_poblacion"] = np.log1p(
    df["poblacion_2025"]
)

In [162]:
scaler = MinMaxScaler()

df["score_superficie"] = scaler.fit_transform(
    df[["superficie_residencial_m2"]]
)

df["score_log_poblacion"] = scaler.fit_transform(
    df[["log_poblacion"]]
)

df["score_suelo_hab"] = scaler.fit_transform(
    df[["suelo_por_habitante"]]
)

In [163]:
df["coopscore_v6"] = (

    df["score_superficie"] * 0.35 +

    df["score_log_poblacion"] * 0.35 +

    df["score_suelo_hab"] * 0.10 +

    df["score_capital_norm"] * 0.10 +

    df["score_precio"] * 0.05 +

    df["score_litoral"] * 0.05

) * 100

In [164]:
df["IVU"] = (

    df["score_superficie"] * 0.50 +

    df["score_suelo_hab"] * 0.30 +

    df["score_capital_norm"] * 0.20

) * 100

In [165]:
df["IVEC"] = (

    df["score_log_poblacion"] * 0.40 +

    df["score_precio"] * 0.40 +

    df["score_litoral"] * 0.20

) * 100

In [166]:
df["perfil_municipio"] = "Equilibrado"

In [167]:
df.loc[
    (df["IVU"] >= 60) &
    (df["IVEC"] < 40),
    "perfil_municipio"
] = "Potencial Urbanístico"

In [168]:
df.loc[
    (df["IVEC"] >= 60) &
    (df["IVU"] < 40),
    "perfil_municipio"
] = "Potencial Económico"

In [169]:
df.loc[
    (df["IVU"] >= 50) &
    (df["IVEC"] >= 50),
    "perfil_municipio"
] = "Alta Prioridad"

In [170]:
df[
    [
        "municipio",
        "coopscore_v6",
        "IVU",
        "IVEC",
        "perfil_municipio"
    ]
].sort_values(
    "coopscore_v6",
    ascending=False
).head(20)

,municipio,coopscore_v6,IVU,IVEC,perfil_municipio
0,"Gabias, Las",76.121955,72.384937,42.740710,Equilibrado
2,Atarfe,72.756889,67.958269,45.690729,Equilibrado
1,Guadix,69.512419,64.509596,36.342431,Potencial Urbanístico
4,Albolote,68.868335,62.137223,48.994165,Equilibrado
6,Salobreña,65.214732,50.520668,74.859907,Alta Prioridad
5,Loja,63.949539,55.307212,36.006203,Equilibrado
9,Armilla,62.162024,48.183518,59.216023,Equilibrado
12,Almuñécar,61.926164,34.620780,94.535168,Potencial Económico
3,Benamaurel,60.685000,81.663408,26.154654,Potencial Urbanístico
11,Churriana de la Vega,57.640176,46.293561,52.522654,Equilibrado


In [171]:
df["perfil_municipio"].value_counts()

perfil_municipio
Equilibrado              123
Potencial Urbanístico      2
Potencial Económico        2
Alta Prioridad             1
Name: count, dtype: int64

In [172]:
df.to_csv(
    DATA_PROCESSED / "dataset_final_tfm_v7.csv",
    index=False
)

print("Dataset final generado")

Dataset final generado
